# G'Contest 2026 - Fraud & Anomaly Detection

Notebook này dùng dữ liệu thật trong `Processed_Data/`. Dữ liệu không có nhãn fraud đã xác minh, vì vậy bài làm không tạo synthetic label. Mục tiêu là xây dựng framework phát hiện bất thường, tạo weak label từ rule nghiệp vụ, huấn luyện prevention model, xếp hạng giao dịch cần can thiệp và giải thích nguyên nhân theo nghiệp vụ ngân hàng.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from fraud_pipeline import PipelineConfig, run_pipeline, load_reference_tables, aggregate_activity, validate_schema

RAW_DIR = PROJECT_ROOT / 'Processed_Data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
sns.set_theme(style='whitegrid')

## 2. Assignment Interpretation

Theo assignment, fraud track cần kết hợp transaction metadata với digital footprint để:

- xây behavioral baseline cho từng khách hàng,
- phát hiện account takeover và unauthorized transfers,
- mở rộng sang money laundering patterns nếu dữ liệu cho phép,
- xuất được human-readable reason cho từng dự báo.

Do không có confirmed fraud label, notebook tạo weak label từ root-cause rules, rồi huấn luyện supervised prevention model để quyết định Allow / Monitor / Step-up / Block.

## 2.1 Final 4-Phase Flow

**Phase 1 - Data Foundation & Feature Engineering**

1. Data Cleaning: chuẩn hóa schema, xử lý beneficiary/merchant, aggregate activity log, ghép product snapshots.
2. Four Baseline Metrics: Transactional, Financial, Environmental, Behavioral.
3. Customer 360 Feature Extraction: rolling window 30/60/90 ngày và baseline một dòng cho mỗi khách hàng.

**Phase 2 - EDA, Thresholding & Auto-Labeling**

4. Baseline EDA: lift chart, heatmap, network exposure, Customer 360 risk map.
5. IQR Thresholding: `Q3 + 1.5 * IQR`.
6. Dynamic Rule Engine: Account Takeover, Unauthorized Transfer, AML/Mule Network.
7. Risk-Scoring & Auto-Labeling: tạo `rule_fraud_label` weak supervision.

**Phase 3 - Machine Learning & Hybrid Check**

8. ML Training: RandomForest học weak label, theo dõi recall, precision và false-positive rate.
9. Hybrid Matrix: Rule+ML = Block/Hold, Rule-only = Step-up/eKYC, ML-only = Watchlist, No-alert = Allow.

**Phase 4 - Deployment, Dashboard & xAI**

10. Business Dashboard: prevention impact, protected amount, Customer 360, case review.
11. xAI: reason codes + SHAP surrogate giải thích final hybrid prevention score.

## 3. Data Overview And Schema Check

In [ ]:
tables = load_reference_tables(RAW_DIR)
row_counts = {name: len(df) for name, df in tables.items()}
row_counts

In [ ]:
# Activity table is large, so the production pipeline aggregates it by customer-date.
# Run this cell if you want a fresh schema report before the full pipeline.
activity_daily, activity_no_threshold = aggregate_activity(RAW_DIR, chunksize=1_000_000)
schema_report = validate_schema(RAW_DIR, tables, activity_daily)
pd.DataFrame(schema_report).T

## 4. Cause-First Fraud Hypotheses

BGK có xu hướng đánh giá cao việc phân tích nguyên nhân trước. Framework gom tín hiệu thành ba nhánh:

1. **Account takeover / identity compromise**: thiết bị mới, IP mới, hoạt động đêm, người thụ hưởng mới, late-stage activity.
2. **Unauthorized transfer / capital outflow**: chuyển khoản ra ngoài, số tiền lệch baseline, daily burst, dòng tiền ra lớn so với CASA.
3. **AML network / mule-account pattern**: IP/device dùng chung, beneficiary nhận tiền từ nhiều khách hàng, giao dịch số tròn giá trị cao.

`ACTIVITY_NO` được dùng như thứ tự hành động: số lớn hơn là bước sau hơn. Pipeline lấy top 10% `ACTIVITY_NO` làm late-stage activity dựa trên phân phối thật.

In [ ]:
pd.DataFrame({
    'root_cause_branch': [
        'Account takeover / identity compromise',
        'Unauthorized transfer / capital outflow',
        'AML network / mule-account pattern',
    ],
    'signals': [
        'New device/IP, night access, new beneficiary, late-stage activity',
        'Outside-bank transfer, high amount vs customer baseline, daily burst, cash-out vs CASA',
        'Shared device/IP/beneficiary, round high-value transfers, repeated external transfers',
    ],
    'business_action': [
        'Step-up authentication and account verification',
        'Manual review / temporary hold for high-risk transfer',
        'AML escalation and network investigation',
    ],
})

## 5. Mentor Feedback Incorporated

- `Beneficiary_CUSTOMER_NUMBER = 0/0.0/NaN` không bị coi mặc định là missing data; pipeline tách nhóm này thành merchant/non-customer beneficiary và phân tích tiếp bằng `Merchant_ID_Masked` + loại giao dịch.
- Nhóm không có customer beneficiary không được đưa vào mạng lưới customer-to-customer money mule.
- Nếu merchant nội bộ/tín dụng không có customer beneficiary nhưng đi cùng tín hiệu bất thường, pipeline đưa vào reason code để review.
- Overdue lending/credit được quy về nhóm rủi ro 1-5 để làm bối cảnh khách hàng, nhưng không dùng một mình để kết luận fraud.
- Baseline hiện được đóng gói thành `customer_360_baseline.csv` với 4 nhóm Transactional, Financial, Environmental, Behavioral.
- Threshold chính có thêm ngưỡng IQR cá nhân hóa `Q3 + 1.5*IQR` và rolling window 30/60/90 ngày.
- Hành động cuối dùng hybrid decision matrix: Rule+ML = Block/Hold, Rule-only = Step-up/eKYC, ML-only = Watchlist, no-alert = Allow.
- Vì title là giảm thiểu rủi ro, mục tiêu tối ưu ưu tiên recall/prevention coverage; false positive rate vẫn được báo để kiểm soát trải nghiệm khách hàng.

## 6. Run Full Pipeline

In [ ]:
config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    figures_dir=OUTPUT_DIR / 'figures',
    report_dir=PROJECT_ROOT / 'report',
)
metrics = run_pipeline(config)
metrics['review_queue']

## 7. Risk Outputs

In [ ]:
risk = pd.read_csv(OUTPUT_DIR / 'transaction_risk_scores.csv')
customers = pd.read_csv(OUTPUT_DIR / 'customer_risk_summary.csv')
customer_360 = pd.read_csv(OUTPUT_DIR / 'customer_360_baseline.csv')
root_cause = pd.read_csv(OUTPUT_DIR / 'root_cause_summary.csv')
insights = pd.read_csv(OUTPUT_DIR / 'insight_summary.csv')
risk.head()

In [ ]:
risk['risk_band'].value_counts().reindex(['Low','Medium','High','Critical'])

## 8. Customer 360 Baseline

Mỗi khách hàng là một dòng, mô tả 'DNA tài chính' hiện tại bằng 4 nhóm baseline: Transactional, Financial, Environmental và Behavioral. Rolling window 30/60/90 ngày giúp baseline phản ánh hành vi gần đây hơn thay vì đóng băng theo toàn bộ lịch sử.

In [ ]:
customer_360.head()

In [ ]:
customer_360[[
    'transactional_iqr_upper_amount', 'rolling_30d_txn_count', 'rolling_90d_amount_avg',
    'financial_worst_credit_risk_group', 'environmental_trusted_device_count',
    'behavioral_avg_daily_activity_count'
]].describe().T

## 9. Monthly / Quarterly Stability Backtest

In [ ]:
monthly = pd.read_csv(OUTPUT_DIR / 'monthly_stability.csv')
quarterly = pd.read_csv(OUTPUT_DIR / 'quarterly_stability.csv')
monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(monthly['month'], monthly['avg_risk_score'], marker='o', label='Avg risk score')
ax1.set_ylabel('Avg risk score')
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['high_critical_rate']*100, marker='s', color='#C46243', label='High/Critical rate')
ax2.set_ylabel('High/Critical rate (%)')
plt.title('Monthly stability backtest on 2019 data')
plt.tight_layout()

## 10. Data-Driven EDA Insights

Các insight này được sinh từ scored population sau khi pipeline đọc dữ liệu thật. Mục tiêu là chứng minh framework không chỉ vẽ biểu đồ đơn giản, mà tìm được mối liên hệ giữa baseline, root cause, hybrid decision và business action.

In [ ]:
insights

In [ ]:
from IPython.display import Image, display
for filename in insights['linked_figure'].dropna().unique():
    path = OUTPUT_DIR / 'figures' / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))

## 11. Explainability Examples

In [ ]:
risk[[
    'transaction_row_id', 'CUSTOMER_NUMBER', 'TRANS_DATE', 'TRANS_HOUR', 'TRANS_LV1', 'TRANS_LV2',
    'TRANS_AMOUNT', 'rule_score_0_100', 'model_fraud_probability', 'risk_score_0_100',
    'risk_band', 'hybrid_decision', 'primary_cause_branch', 'top_reasons', 'recommended_action'
]].head(10)

## 12. Hybrid Decision Matrix And Risk Band Policy

Ma trận lai giúp biến Rule + ML thành hành động thực tế. Đây là phần quan trọng để bài không chỉ dừng ở phát hiện, mà chuyển sang prevention operation.

In [ ]:
pd.Series(metrics['risk_band_policy'])

In [ ]:
risk['hybrid_decision'].value_counts()

In [ ]:
pd.crosstab(risk['hybrid_decision'], risk['prevention_action'])

## 13. SHAP xAI Surrogate

In [ ]:
# Run once after the main pipeline if SHAP files do not exist:
# !python src/xai_shap_engine.py
shap_importance = pd.read_csv(OUTPUT_DIR / 'shap_feature_importance.csv')
shap_local = pd.read_csv(OUTPUT_DIR / 'shap_local_explanations.csv')
shap_importance.head(15)

In [ ]:
shap_local.head(10)

## 14. Evaluation With Weak Labels

Vì dữ liệu không có confirmed fraud label, các metric dưới đây được đo theo `rule_fraud_label`. Đây là weak label sinh từ rule nghiệp vụ, không phải nhãn investigator xác nhận. Cách trình bày đúng là: framework hiện đo khả năng model học lại và vận hành hóa rule engine, đồng thời chuẩn bị feedback loop để thay weak label bằng confirmed label sau này.

Theo feedback mentor, fraud track ưu tiên bắt được nhiều case rủi ro nhất. Vì vậy cần nhìn recall/prevention coverage trước, sau đó kiểm soát precision và false-positive rate để không làm phiền khách hàng thường.

In [ ]:
with open(OUTPUT_DIR / 'model_metrics.json', encoding='utf-8') as f:
    metrics = json.load(f)
pd.Series(metrics['risk_band_distribution'])

In [ ]:
pd.Series(metrics['supervised_model_metrics'])

In [ ]:
pd.DataFrame(metrics['supervised_model_metrics']['validation_confusion_matrix_at_high_threshold'], index=[0])

In [ ]:
pd.DataFrame(metrics['top_surrogate_features'].items(), columns=['feature', 'importance']).head(15)

## 15. Report Figures

In [ ]:
for path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 16. Live Demo And Slide Deck

Sau khi chạy pipeline, có thể demo bằng CLI hoặc Streamlit:

```bash
python src/customer_risk_advisor.py --top-critical 3
python src/customer_risk_advisor.py --customer-id <CUSTOMER_NUMBER>
python src/customer_risk_advisor.py --transaction-id <transaction_row_id>
streamlit run src/demo_app.py
```

Tạo slide deck PDF/PPTX:

```bash
python src/build_slide_deck.py
```

Demo trả về risk band, nguyên nhân chính, reason codes, SHAP evidence và recommended action bằng ngôn ngữ dễ hiểu cho risk officer.